<section>
<h2>1. Credentials and environment setup</h2>
<p>Before running the examples, create an API key in the <a href="https://console.anthropic.com/settings/keys">Anthropic Console</a>. Create a GitHub fine-grained personal access token (PAT) under GitHub <strong>Settings → Developer settings → Personal access tokens → Fine-grained tokens</strong>. Give it an expiration, restrict it to the repository or repositories needed, and grant only the required permissions. The final example lists commits, so repository <code>Contents: read</code> (and the automatically required metadata access) is sufficient; grant additional permissions only for operations you intend to run.</p>
<p>The first setup cell prompts for a missing <code>ANTHROPIC_API_KEY</code> using hidden input and keeps it only in the current Python kernel. The GitHub MCP example later prompts for <code>GITHUB_PERSONAL_ACCESS_TOKEN</code> only when needed; set that environment variable in advance to skip the prompt. Do not write secrets into notebook source, print them, commit them, or share them. Revoke and replace a key immediately if it is exposed.</p>
<p><strong>Loop primitives:</strong> this cell has no loops; the <code>if</code> guards avoid prompting when a credential is already available.</p>
</section>

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

<section>
<h2>2. Imports and Anthropic client</h2>
<p>Import the Anthropic SDK and subprocess support used by the agent examples. <code>time</code> is used by the bounded retry example. <code>anthropic.Anthropic()</code> reads <code>ANTHROPIC_API_KEY</code> from the environment populated by the setup cell; no key belongs in this code.</p>
<p><strong>Loop primitives:</strong> this cell contains no loops.</p>
</section>

In [ ]:
import anthropic
import subprocess
import time

# Initialize the client (automatically reads ANTHROPIC_API_KEY from your environment variables)
client = anthropic.Anthropic()

<section>
<h2>3. ReAct agent: normalize messages and run a shell tool</h2>
<p>The helper's <code>for msg in messages</code> visits each conversation message once. Its list comprehensions partition content blocks into thinking and non-thinking blocks; <code>result.append</code> builds a normalized transcript without mutating unrelated messages. The outer <code>while True</code> repeats the model → tool → result cycle. A <code>next(...)</code> generator expression selects the first tool-use block. If the model returns no tool call, the agent prints its final text and <code>break</code>s; otherwise it runs the requested command and appends the result for the next turn.</p>
<p>This is a live shell execution example: only run it in a trusted environment and review commands before granting an agent access to sensitive files or credentials.</p>
</section>

In [36]:
def _is_anthropic_thinking_block(block) -> bool:
    """Check if a content block is a thinking-type block."""
    if not isinstance(block, dict):
        return False
    return block.get("type") in ("thinking", "redacted_thinking")


def _reorder_anthropic_thinking_blocks(messages: list[dict]) -> list[dict]:
    """Reorder thinking blocks so they are not the final block in assistant messages.

    This is an Anthropic API requirement: thinking blocks must come before other blocks.
    """
    result = []
    for msg in messages:
        if msg.get("role") == "assistant" and isinstance(msg.get("content"), list):
            content = msg["content"]
            thinking_blocks = [b for b in content if _is_anthropic_thinking_block(b)]
            if thinking_blocks:
                other_blocks = [b for b in content if not _is_anthropic_thinking_block(b)]
                if other_blocks:
                    msg = {**msg, "content": thinking_blocks + other_blocks}
                else:
                    msg = {**msg, "content": thinking_blocks + [{"type": "text", "text": ""}]}
        result.append(msg)
    return result
      

system_prompt = (
    "You are an autonomous system administration agent. You have access to a `run_bash` tool. "
    "You can run multiple commands sequentially if needed to solve the user's request. "
    "Once you have achieved the goal, provide a final textual summary to the user."
)

messages = [
    {
        "role": "user", 
        "content": "Find all Python files (.py) modified in the last 7 days, count them, and show their total disk usage."
    }
]

tools = [
    {
        "name": "run_bash",
        "description": "Execute a shell command locally. Use this tool when command-line operations are required.",
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "The single executable bash command to run."
                }
            },
            "required": ["command"]
        }
    }
]


while True:
    sanitized_messages = _reorder_anthropic_thinking_blocks(messages)
    
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=512,
        system=system_prompt,
        messages=sanitized_messages,
        tools=tools
    )
    
    messages.append({"role": "assistant", "content": response.content})
    
    tool_use = next((b for b in response.content if getattr(b, "type", None) == "tool_use"), None)
    
    if not tool_use:
        # Safely extract text from any text blocks
        final_text = "".join([b.text for b in response.content if hasattr(b, "text")])
        print(final_text)
        ### NO TOOL CALL
        break
        
    cmd = tool_use.input["command"]
    print(f"Running: {cmd}")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    output = res.stdout if res.returncode == 0 else res.stderr
    
    messages.append({
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": output or "[Command executed with no output]"
        }]
    })

Running: find . -name "*.py" -type f -mtime -7 2>/dev/null | tee /tmp/recent_py.txt | wc -l; echo "--- files ---"; cat /tmp/recent_py.txt; echo "--- total size ---"; if [ -s /tmp/recent_py.txt ]; then tr '\n' '\0' < /tmp/recent_py.txt | du -ch --files0-from=- 2>/dev/null | tail -1; else echo "0"; fi
Only one Python file was modified in the last 7 days, and it uses 4.0K of disk space.

- **Count:** 1
- **File:** `./swe.py`
- **Total disk usage:** 4.0K

I searched recursively from the current working directory only. If you meant a different location, such as your home directory or a project root, tell me which path and I'll rerun the search there.


<section>
<h2>4. Compact ReAct agent</h2>
<p>This shorter version repeats the model/tool cycle with a <code>while True</code> loop. The generator expression passed to <code>next</code> finds the first tool-use block, and the final-text list comprehension gathers text blocks. A missing tool call means the agent is finished, so <code>break</code> exits; otherwise the command runs and its result is appended to the conversation.</p>
</section>

In [42]:
def _is_anthropic_thinking_block(block) -> bool:
    """Check if a content block is a thinking-type block."""
    if not isinstance(block, dict):
        return False
    return block.get("type") in ("thinking", "redacted_thinking")

def _reorder_anthropic_thinking_blocks(messages: list[dict]) -> list[dict]:
    """Reorder thinking blocks so they are not the final block in assistant messages."""
    result = []
    for msg in messages:
        if msg.get("role") == "assistant" and isinstance(msg.get("content"), list):
            content = msg["content"]
            thinking_blocks = [b for b in content if _is_anthropic_thinking_block(b)]
            if thinking_blocks:
                other_blocks = [b for b in content if not _is_anthropic_thinking_block(b)]
                if other_blocks:
                    msg = {**msg, "content": thinking_blocks + other_blocks}
                else:
                    msg = {**msg, "content": thinking_blocks + [{"type": "text", "text": ""}]}
        result.append(msg)
    return result

system_prompt = (
    "You are an autonomous ReAct agent. Reason about the task step-by-step, "
    "use `run_bash` to act, observe results, and conclude when finished."
)

tools = [{
    "name": "run_bash",
    "description": "Run bash command.",
    "input_schema": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"]
    }
}]

messages = [{"role": "user", "content": "Find all Python files modified in the last 7 days and count them."}]

while True:
    sanitized_messages = _reorder_anthropic_thinking_blocks(messages)
    
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=512,
        system=system_prompt,
        messages=sanitized_messages,
        tools=tools
    )
    
    messages.append({"role": "assistant", "content": response.content})
    
    tool_use = next((b for b in response.content if getattr(b, "type", None) == "tool_use"), None)
    
    if not tool_use:
        final_text = "".join([b.text for b in response.content if hasattr(b, "text")])
        print(final_text)
        break
        
    cmd = tool_use.input["command"]
    print(f"[Action] Running: {cmd}")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    output = res.stdout if res.returncode == 0 else res.stderr
    
    messages.append({
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": output or "[No output]"
        }]
    })

[Action] Running: find . -name "*.py" -type f -mtime -7 2>/dev/null | tee /tmp/recent_py.txt; echo "COUNT: $(wc -l < /tmp/recent_py.txt)"
There are **2** Python files modified in the last 7 days, both in the current directory:

- `./swe.py`
- `./hello.py`

I searched recursively from the current directory using modification time (`-mtime -7`). The search doesn't cover files outside this directory tree, and it uses modification time, not creation time.


<section>
<h2>5. Test-and-repair agent</h2>
<p>The <code>while True</code> loop requests a command, executes it, and returns either its output or an error to the model. The <code>if/else</code> branches on the command's exit code: failure is fed back for repair, while success is reported as passed. When the model is done, a list comprehension gathers text blocks for the final answer and <code>break</code> exits the loop. This is a model-guided repair cycle, not an automatic local retry loop.</p>
</section>

In [40]:
system_prompt = "You are a code generation and testing agent. Generate a bash command, test it, and if it errors, analyze the traceback/stderr to repair and retry."
tools = [{"name": "run_bash", "description": "Execute command to test it.", "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}}]

messages = [{"role": "user", "content": "Create a python script that prints 'hello' and run it."}]

while True:
    response = client.messages.create(model="claude-haiku-5-5", max_tokens=512, system=system_prompt, messages=messages, tools=tools)
    messages.append({"role": "assistant", "content": response.content})
    
    tool_use = next((b for b in response.content if getattr(b, "type", None) == "tool_use"), None)
    if not tool_use:
        print("".join([b.text for b in response.content if hasattr(b, "text")]))
        break
        
    cmd = tool_use.input["command"]
    print(f"[Test Phase] Executing: {cmd}")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    
    if res.returncode != 0:
        output = f"ERROR (Exit Code {res.returncode}):\n{res.stderr}"
        print(f"[Repair Phase] Test failed. Feeding error back for repair.")
    else:
        output = f"SUCCESS:\n{res.stdout}"
        print(f"[Test Phase] Passed successfully.")
        
    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_use.id, "content": output}]})

[Test Phase] Executing: printf 'print("hello")\n' > hello.py && python3 hello.py
[Test Phase] Passed successfully.
I created `hello.py` containing `print("hello")` and ran it with `python3`. It printed `hello` and exited successfully.


<section>
<h2>6. Plan-and-execute agent</h2>
<p>The <code>while True</code> loop lets the model request successive plan steps through the shell tool. Each command result is appended to the conversation so the model can choose the next step. When the model stops requesting tools, a list comprehension gathers response text, it is printed, and <code>break</code> finishes the loop.</p>
</section>

In [41]:
system_prompt = (
    "First, output a clear numbered plan to solve the objective. "
    "Then, use the `run_bash` tool to execute each step of the plan sequentially until complete."
)
tools = [{"name": "run_bash", "description": "Execute step.", "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}}]

messages = [{"role": "user", "content": "Backup the current directory into a tar.gz file and check its size."}]

while True:
    response = client.messages.create(model="claude-haiku-5-5", max_tokens=1024, system=system_prompt, messages=messages, tools=tools)
    messages.append({"role": "assistant", "content": response.content})
    
    tool_use = next((b for b in response.content if getattr(b, "type", None) == "tool_use"), None)
    if not tool_use:
        print("".join([b.text for b in response.content if hasattr(b, "text")]))
        break
        
    cmd = tool_use.input["command"]
    print(f"[Plan Execution] Running step: {cmd}")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    output = res.stdout if res.returncode == 0 else res.stderr
    
    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_use.id, "content": output or "[Step completed]"}]})

[Plan Execution] Running step: pwd && echo "---" && ls -la && echo "---" && du -sh . 2>/dev/null
[Plan Execution] Running step: cd .. && tar -czf "swe_backup_$(date +%Y%m%d_%H%M%S).tar.gz" -C . swe && ls -l swe_backup_*.tar.gz
[Plan Execution] Running step: cd .. && f=$(ls -t swe_backup_*.tar.gz | head -1) && echo "Archive: $(pwd)/$f" && du -h "$f" && echo "--- integrity test ---" && tar -tzf "$f" > /dev/null && echo "OK" && echo "--- contents ---" && tar -tzvf "$f"
The backup is done. The `swe` directory is archived as a compressed tarball and the archive passes an integrity check.

- **Archive:** `/home/jphardee/Agency/cupids-harness/swe_backup_20261030_075935.tar.gz`
- **Size:** 8.0K compressed, down from 44K on disk for the original directory.
- **Contents:** all 7 files, including the `.ipynb_checkpoints` folder, stored under `swe/`.

I wrote the archive to the parent directory rather than inside `swe/`, so it doesn't end up including itself.


<section>
<h2>7. Bounded local retries</h2>
<p>The outer <code>while True</code> loop handles model/tool turns. Inside it, <code>for attempt in range(1, MAX_LOCAL_RETRIES + 1)</code> makes a finite number of local command attempts. Success stores stdout and <code>break</code>s immediately; failure reports the attempt, waits one second with <code>time.sleep</code>, and preserves the final stderr for the model. If all attempts fail, the loop ends naturally at its configured limit. A no-tool model response uses a list comprehension to gather final text, then exits the outer loop.</p>
</section>

In [43]:
system_prompt = "You are a robust sysadmin agent. Use `run_bash` to execute commands."
tools = [{"name": "run_bash", "description": "Run bash command.", "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}}]

messages = [{"role": "user", "content": "Check disk space usage."}]
MAX_LOCAL_RETRIES = 3

while True:
    response = client.messages.create(model="claude-haiku-5-5", max_tokens=512, system=system_prompt, messages=messages, tools=tools)
    messages.append({"role": "assistant", "content": response.content})
    
    tool_use = next((b for b in response.content if getattr(b, "type", None) == "tool_use"), None)
    if not tool_use:
        print("".join([b.text for b in response.content if hasattr(b, "text")]))
        break
        
    cmd = tool_use.input["command"]
    print(f"Running command: {cmd}")
    
    # Local Multi-Attempt Retry Wrapper
    output = ""
    for attempt in range(1, MAX_LOCAL_RETRIES + 1):
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
        if res.returncode == 0:
            output = res.stdout
            break
        else:
            print(f"[Attempt {attempt}/{MAX_LOCAL_RETRIES}] Failed. Retrying...")
            time.sleep(1)
            output = f"Command failed after {MAX_LOCAL_RETRIES} attempts. Stderr: {res.stderr}"

    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_use.id, "content": output or "[No output]"}]})

Running command: df -h
Disk usage on the system:

| Filesystem | Size | Used | Avail | Use% | Mounted on |
|---|---|---|---|---|---|
| `/dev/nvme0n1p2` | 916G | 754G | 116G | **87%** | `/` |
| `/dev/nvme0n1p1` | 511M | 6.2M | 505M | 2% | `/boot/efi` |
| `tmpfs` | 16G | 245M | 16G | 2% | `/dev/shm` |
| `tmpfs` | 3.1G | 2.0M | 3.1G | 1% | `/run` |
| `tmpfs` | 3.1G | 240K | 3.1G | 1% | `/run/user/1000` |
| `tmpfs` | 5.0M | 20K | 5.0M | 1% | `/run/lock` |
| `efivarfs` | 128K | 66K | 58K | 54% | `/sys/firmware/efi/efivars` |

The root filesystem is the only one that needs attention. It is 87% full with 116G free, which is fine for now but will fill up if usage keeps growing. The other filesystems are nearly empty.

I can find the largest directories on `/` if you want to see what is using the space.


<section>
<h2>8. Branching tool exploration</h2>
<p>The outer <code>while True</code> loop requests candidates from the model. A list comprehension collects <em>all</em> tool-use blocks, and <code>for tool_use in tool_uses</code> executes each candidate and accumulates its result before one user message is appended. Despite supporting multiple candidates in one response, this code runs them sequentially—not in parallel. If no candidates are returned, a list comprehension gathers the final answer, which is printed before <code>break</code> ends the loop.</p>
</section>

In [45]:
system_prompt = (
    "You are a tree-search agent. If a standard command fails or has multiple interpretations, "
    "propose alternative branches using `run_bash` to test different strategies."
)

tools = [{
    "name": "run_bash",
    "description": "Test command variant.",
    "input_schema": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"]
    }
}]

messages = [{"role": "user", "content": "Find python files using either `find` or `fd` command."}]

while True:
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=512,
        system=system_prompt,
        messages=messages,
        tools=tools
    )
    
    # 1. Record assistant response
    messages.append({"role": "assistant", "content": response.content})
    
    # 2. Extract ALL tool use blocks (supports multi-tool parallel calls / branching)
    tool_uses = [b for b in response.content if getattr(b, "type", None) == "tool_use"]
    
    if not tool_uses:
        # No tools called -> final text answer reached
        final_text = "".join([b.text for b in response.content if hasattr(b, "text")])
        print(final_text)
        break
        
    # 3. Execute all requested tool commands and collect their results
    tool_results = []
    for tool_use in tool_uses:
        cmd = tool_use.input["command"]
        print(f"[Branch Exploration] Testing candidate command: {cmd}")
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
        
        if res.returncode == 0:
            output = f"[Branch Successful]\n{res.stdout}"
        else:
            output = f"[Branch Failed with error]: {res.stderr}. Try another approach."
            
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": output or "[Command executed with no output]"
        })
        
    # 4. Append ALL tool results back to conversation history in a single user message
    messages.append({
        "role": "user",
        "content": tool_results
    })

[Branch Exploration] Testing candidate command: find . -type f -name "*.py" 2>/dev/null
[Branch Exploration] Testing candidate command: fd -e py 2>/dev/null
Using `find`, I found two Python files in the current directory:

- `./swe.py`
- `./hello.py`

The `fd` command failed with the error "Branch Failed with error: . Try another approach." The error doesn't say why. `fd` may not be installed, or the command may have hit an issue. The `find` result is complete, so I didn't retry `fd`. If you want to check it, run `command -v fd` to see whether it's installed.


<section>
<h2>9. GitHub MCP agent</h2>
<p>This example connects to the GitHub MCP server with the PAT from the setup cell, discovers its tools, then uses the Anthropic API to choose actions. The helper uses a <code>for</code> loop and list comprehensions to normalize messages and convert discovered tools. The agent's <code>while True</code> loop repeats model/tool turns; a list comprehension identifies tool-use blocks, and <code>for tool_use in tool_uses</code> invokes every requested MCP tool. A nested <code>for content_block in result.content</code> gathers textual output. These tool calls are awaited one at a time, so execution is sequential. A final-text list comprehension gathers the answer, and <code>break</code> exits when the model returns no tools. The <code>async with</code> blocks manage the MCP process/session lifetime; they are context managers, not loops.</p>
<p>The final cell checks that a GitHub token is present instead of embedding a placeholder. Keep the token restricted to the required repositories and permissions. Review requested tool inputs before using this agent for write operations.</p>
<h3>Install the GitHub MCP server with npm</h3>
<p>With Node.js installed, run <code>npm install -g @modelcontextprotocol/server-github</code>. This provides the <code>mcp-server-github</code> executable used below.</p>
</section>

In [ ]:
import asyncio
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def _is_anthropic_thinking_block(block) -> bool:
    if not isinstance(block, dict):
        return False
    return block.get("type") in ("thinking", "redacted_thinking")

def _reorder_anthropic_thinking_blocks(messages: list[dict]) -> list[dict]:
    result = []
    for msg in messages:
        if msg.get("role") == "assistant" and isinstance(msg.get("content"), list):
            content = msg["content"]
            thinking_blocks = [b for b in content if _is_anthropic_thinking_block(b)]
            if thinking_blocks:
                other_blocks = [b for b in content if not _is_anthropic_thinking_block(b)]
                if other_blocks:
                    msg = {**msg, "content": thinking_blocks + other_blocks}
                else:
                    msg = {**msg, "content": thinking_blocks + [{"type": "text", "text": ""}]}
        result.append(msg)
    return result

async def main():
    if not os.environ.get("GITHUB_PERSONAL_ACCESS_TOKEN"):
        os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = getpass("GitHub fine-grained PAT: ")
    nvm_bin_dir = "/home/jphardee/.nvm/versions/node/v26.13.0/bin"
    current_path = os.environ.get("PATH", "")

    subprocess_env = {
      **os.environ,
      "PATH": f"{nvm_bin_dir}:{current_path}",
    }
    server_params = StdioServerParameters(
      command=f"{nvm_bin_dir}/mcp-server-github",
      args=[],
      env=subprocess_env,
    )

    # 2. Establish connection to the MCP Server via stdio
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize the MCP handshake
            await session.initialize()
            
            # 3. Discover available tools from the GitHub MCP server
            mcp_tools_response = await session.list_tools()

            anthropic_tools = [
                {
                    "name": tool.name,
                    "description": tool.description,
                    "input_schema": (
                        tool.input_schema
                    ),
                }
                for tool in mcp_tools_response.tools
            ]
            print(f"Successfully loaded {len(anthropic_tools)} tools from GitHub MCP Server.")

            system_prompt = (
                "You are an autonomous GitHub management agent. You have access to GitHub tools via MCP. "
                "Use them to solve the user's repository requests, then provide a final textual summary."
            )

            messages = [
                {
                    "role": "user",
                    "content": "List the last 5 commits in my repository 'https://github.com/jph6366/cupids-harness'."
                }
            ]

            # 4. Run the standard agent loop
            while True:
                sanitized_messages = _reorder_anthropic_thinking_blocks(messages)
                
                response = client.messages.create(
                    model="claude-haiku-5-5",
                    max_tokens=1024,
                    system=system_prompt,
                    messages=sanitized_messages,
                    tools=anthropic_tools
                )
                
                messages.append({"role": "assistant", "content": response.content})
                
                # Check for tool usage
                tool_uses = [b for b in response.content if getattr(b, "type", None) == "tool_use"]
                
                if not tool_uses:
                    final_text = "".join([b.text for b in response.content if hasattr(b, "text")])
                    print(f"\n[Agent Final Answer]:\n{final_text}")
                    break
                
                # Execute tool calls through the active MCP session
                tool_results = []
                for tool_use in tool_uses:
                    print(f"\n[Calling MCP Tool]: {tool_use.name} with input {tool_use.input}")
                    try:
                        # Invoke tool remotely through the MCP server subprocess session
                        result = await session.call_tool(tool_use.name, arguments=tool_use.input)
                        
                        # Extract content blocks from MCP result
                        output_content = ""
                        for content_block in result.content:
                            if hasattr(content_block, "text"):
                                output_content += content_block.text
                        
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": output_content or "[Tool executed with no output]"
                        })
                    except Exception as e:
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": f"Error executing tool: {str(e)}"
                        })
                
                messages.append({
                    "role": "user",
                    "content": tool_results
                })

                

await main()